# Getting the Bronze Data From External storage

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
import builtins
from pyspark.sql.functions import current_date, lit
base_path = "abfss://orders@adlsstdout001tgt.dfs.core.windows.net/Orders/"
import builtins
from pyspark.sql.functions import current_date, lit

# ---- Year ----
folders = dbutils.fs.ls(base_path)
latest_year = builtins.max([f.name.replace("/", "") for f in folders])

# ---- Month ----
folders = dbutils.fs.ls(f"{base_path}{latest_year}/")
latest_month = builtins.max([f.name.replace("/", "") for f in folders])

# ---- Day ----
folders = dbutils.fs.ls(f"{base_path}{latest_year}/{latest_month}/")
latest_day = builtins.max([f.name.replace("/", "") for f in folders])

# ---- Final Path ----
latest_path = f"{base_path}{latest_year}/{latest_month}/{latest_day}/"

print(f"Reading from: {latest_path}")

# ---- File Date ----
file_date = f"{latest_year}-{latest_month}-{latest_day}"

# ---- Read parquet ----
raw_df = (
    spark.read
    .parquet(latest_path)
    .withColumn("ingest_date", current_date())
    .withColumn("file_date", lit(file_date))
)

display(raw_df.limit(10))

# Data Cleaning

In [0]:
import builtins
from pyspark.sql.functions import current_date, lit
base_path = "abfss://orders@adlsstdout001tgt.dfs.core.windows.net/Orders/"
import builtins
from pyspark.sql.functions import current_date, lit
from pyspark.sql.window import Window
from pyspark.sql.functions import col
# ==========================================
# STANDARDIZE COLUMN NAMES
# ==========================================

df = (
    raw_df
    .toDF(*[col.lower().strip() for col in raw_df.columns])
)

# ==========================================
# REMOVE DUPLICATES
# BUSINESS KEY = order_id
# KEEP LATEST FILE DATE
# ==========================================

window_spec = Window.partitionBy("order_id").orderBy(col("file_date").desc())

df = (
    df
    .withColumn("row_num", row_number().over(window_spec))
    .filter(col("row_num") == 1)
    .drop("row_num")
)

# ==========================================
# TRIM STRING COLUMNS
# ==========================================

string_cols = [field.name for field in df.schema.fields if isinstance(field.dataType, StringType)]

for c in string_cols:
    df = df.withColumn(c, trim(col(c)))

# ==========================================
# HANDLE EMPTY STRINGS AS NULL
# ==========================================

for c in string_cols:
    df = df.withColumn(
        c,
        when(col(c) == "", None).otherwise(col(c))
    )

# ==========================================
# STANDARDIZE ORDER STATUS
# ==========================================

df = (
    df
    .withColumn("order_status", lower(trim(col("order_status"))))
)

# ==========================================
# VALIDATE ORDER STATUS
# ==========================================

valid_status = [
    "created",
    "approved",
    "invoiced",
    "processing",
    "shipped",
    "delivered",
    "canceled",
    "unavailable"
]

df = (
    df
    .withColumn(
        "order_status",
        when(col("order_status").isin(valid_status), col("order_status"))
        .otherwise("unknown")
    )
)

# ==========================================
# CONVERT TIMESTAMP COLUMNS
# ==========================================

timestamp_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for c in timestamp_cols:
    df = df.withColumn(
        c,
        to_timestamp(col(c))
    )

# ==========================================
# CONVERT DATE COLUMNS
# ==========================================

date_cols = [
    "ingest_date",
    "file_date"
]

for c in date_cols:
    df = df.withColumn(
        c,
        to_date(col(c))
    )

# ==========================================
# DATA QUALITY CHECKS
# ==========================================

# Remove records with null primary key
df = df.filter(col("order_id").isNotNull())

# Remove invalid customer ids
df = df.filter(col("customer_id").isNotNull())

# Remove future purchase dates
df = df.filter(
    col("order_purchase_timestamp") <= current_timestamp()
)

# ==========================================
# BUSINESS VALIDATIONS
# ==========================================

# Approved date should be after purchase date
df = (
    df
    .withColumn(
        "approved_validation",
        when(
            col("order_approved_at") >= col("order_purchase_timestamp"),
            lit("VALID")
        ).otherwise(lit("INVALID"))
    )
)

# Delivery should happen after carrier date
df = (
    df
    .withColumn(
        "delivery_validation",
        when(
            col("order_delivered_customer_date") >= col("order_delivered_carrier_date"),
            lit("VALID")
        ).otherwise(lit("INVALID"))
    )
)

# ==========================================
# DERIVED BUSINESS COLUMNS
# ==========================================

df = (
    df

    # Order Processing Time
    .withColumn(
        "approval_time_hours",
        round(
            (
                unix_timestamp("order_approved_at") -
                unix_timestamp("order_purchase_timestamp")
            ) / 3600,
            2
        )
    )

    # Delivery Time
    .withColumn(
        "delivery_time_days",
        datediff(
            col("order_delivered_customer_date"),
            col("order_purchase_timestamp")
        )
    )

    # Estimated Delay
    .withColumn(
        "delivery_delay_days",
        datediff(
            col("order_delivered_customer_date"),
            col("order_estimated_delivery_date")
        )
    )

    # Delivered On Time Flag
    .withColumn(
        "is_delivered_on_time",
        when(
            col("order_delivered_customer_date") <= col("order_estimated_delivery_date"),
            lit(1)
        ).otherwise(lit(0))
    )

    # Year Month Partition
    .withColumn(
        "order_year_month",
        date_format(col("order_purchase_timestamp"), "yyyy-MM")
    )
)

# ==========================================
# ADD AUDIT COLUMNS
# ==========================================

df = (
    df
    .withColumn("created_date", current_timestamp())
    .withColumn("updated_date", current_timestamp())
    .withColumn("data_source", lit("bronze_orders"))
)

# ==========================================
# REORDER COLUMNS
# ==========================================

final_columns = [
    "order_id",
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "approval_time_hours",
    "delivery_time_days",
    "delivery_delay_days",
    "is_delivered_on_time",
    "approved_validation",
    "delivery_validation",
    "order_year_month",
    "ingest_date",
    "file_date",
    "created_date",
    "updated_date",
    "data_source"
]

df = df.select(*final_columns)
display(df.limit(10))

In [0]:
tablename = "ecom.silver.fact_silver_orders"

In [0]:
(
        df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .option(
            "path",
            "abfss://silver@silverprocessdbstorage.dfs.core.windows.net/Orders"
        )
        .saveAsTable(tablename)
    )

print("Full Load Completed Successfully")